In [1]:
pip install pyscf rdkit pubchempy==1.0.4

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 9.7 MB/s eta 0:00:00
  Created wheel for pubchempy: filename=PubChemPy-1.0.4-py3-none-any.whl size=13819 sha256=004e0583f34f37fc365a71f368b5e060424811c697e42cd5d22a592316912232
  Stored in directory: /root/.cache/pip/wheels/8b/e3/6c/3385b2db08b0985a87f5b117f98d0cb61a3ae3ca3bcbbd8307
Successfully built pubchempy


In [3]:
## Generate molecular structure from SMILES and perform calculations with PySCF

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from pyscf import gto, dft, grad, hessian
import time

# Generate molecule from SMILES
ethanol = Chem.MolFromSmiles('CCO')

# Generate 3D coordinates and optimize with MMFF
ethanol = Chem.AddHs(ethanol)
AllChem.EmbedMolecule(ethanol)
AllChem.MMFFOptimizeMolecule(ethanol)

# Extract coordinates and atom symbols
conf = ethanol.GetConformer()
atom_symbols = []
atom_coords = []
for atom in ethanol.GetAtoms():
    pos = conf.GetAtomPosition(atom.GetIdx())
    atom_symbols.append(atom.GetSymbol())
    atom_coords.append([pos.x, pos.y, pos.z])

# Prepare atom data for PySCF
atom_data = [[symbol, coord] for symbol, coord in zip(atom_symbols, atom_coords)]

# Create PySCF molecule object
mol = gto.Mole()
mol.atom = atom_data
mol.basis = '6-31g(d)'
mol.charge = 0
mol.spin = 0  # singlet
mol.verbose = 4
mol.build()

# Print initial structure
print("Initial molecular structure:")
for symbol, coord in zip(atom_symbols, atom_coords):
    print(f"{symbol} {coord[0]:.6f} {coord[1]:.6f} {coord[2]:.6f}")

# Set up and run DFT calculation
mf = dft.RKS(mol)
mf.xc = 'B3LYP'
mf.kernel()

print(f"\nSCF energy: {mf.e_tot:.8f} Hartree")

# Calculate gradient
g = grad.RKS(mf)
gradients = g.kernel()
grad_norm = np.linalg.norm(gradients)
print(f"Gradient norm: {grad_norm:.6f}")

# Calculate Hessian matrix
hess = hessian.RHF(mf)
h = hess.kernel()

# Convert Hessian from (natm,natm,3,3) to (3*natm,3*natm) format
natm = mol.natm
h_mat = np.zeros((3*natm, 3*natm))
for i in range(natm):
    for j in range(natm):
        for k in range(3):
            for l in range(3):
                h_mat[i*3+k, j*3+l] = h[i, j, k, l]

# Define atomic masses
masses = {"H": 1.008, "C": 12.011, "O": 15.999}
atom_masses = [masses.get(atom[0], 1.0) for atom in mol.atom]

# Create mass matrix
mass_mat = np.zeros((3*natm, 3*natm))
for i, mass in enumerate(atom_masses):
    for j in range(3):
        mass_mat[3*i+j, 3*i+j] = 1.0 / np.sqrt(mass)

# Calculate mass-weighted Hessian
mw_hess = np.dot(np.dot(mass_mat, h_mat), mass_mat)

# Calculate eigenvalues and eigenvectors
eigvals, eigvecs = np.linalg.eigh(mw_hess)

# Convert to frequencies in cm^-1
hartree_to_cm1 = 219474.6
freq_cm1 = np.sign(eigvals) * np.sqrt(np.abs(eigvals)) * hartree_to_cm1

# Print frequency results
print("\nVibrational frequencies (cm-1):")
for i, freq in enumerate(freq_cm1):
    print(f"Mode {i+1}: {freq:.2f} cm-1")

# Check for imaginary frequencies
n_imag = np.sum(freq_cm1 < 0)
if n_imag > 0:
    print(f"\nWarning: {n_imag} imaginary frequencies found.")
    print("This may indicate a transition state or non-minimum structure.")
else:
    print("\nAll frequencies are positive. This structure is stable.")

# Results summary
print("\nCalculation summary:")
print(f"- SCF energy: {mf.e_tot:.8f} Hartree")
print(f"- Gradient norm: {grad_norm:.6f}")
print(f"- Frequency range: {min(freq_cm1):.2f} to {max(freq_cm1):.2f} cm-1")
print("- Method: B3LYP/6-31G(d)")

# Save results to file
with open('ethanol_pyscf_results.txt', 'w') as f:
    f.write("Ethanol PySCF Calculation Results\n\n")
    f.write("Method: B3LYP/6-31G(d)\n\n")

    f.write("Molecular structure (RDKit optimized):\n")
    for symbol, coord in zip(atom_symbols, atom_coords):
        f.write(f"{symbol} {coord[0]:.6f} {coord[1]:.6f} {coord[2]:.6f}\n")

    f.write(f"\nSCF energy: {mf.e_tot:.8f} Hartree\n")
    f.write(f"Gradient norm: {grad_norm:.6f}\n\n")

    f.write("Frequencies (cm-1):\n")
    for i, freq in enumerate(freq_cm1):
        f.write(f"Mode {i+1}: {freq:.2f} cm-1\n")

    if n_imag > 0:
        f.write(f"\nWarning: {n_imag} imaginary frequencies found.\n")
    else:
        f.write("\nAll frequencies are positive. This structure is stable.\n")

print("\nResults saved to 'ethanol_pyscf_results.txt'")

System: uname_result(system='Linux', node='118cf046b967', release='6.1.85+', version='#1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024', machine='x86_64')  Threads 2
Python 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0]
numpy 1.26.4  scipy 1.13.1  h5py 3.12.1
Date: Sun Mar  9 06:21:34 2025
PySCF version 2.8.0
PySCF path  /usr/local/lib/python3.11/dist-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 26
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C      0.888939807574  -0.158324301196  -0.049508470324 AA    1.679852777539  -0.299189568123  -0.093557449759 Bohr   0.0
[INPUT]  2 C     -0.464880812592   0.475661545210   0.193920295030 AA   -0.878497416364   0.898870048435   0.366456247602 Bohr   0.0

In [ ]:
## Program to perform quantum chemistry calculations from SMILES string using PySCF

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from pyscf import gto, dft, grad, hessian
import time

def calculate_with_pyscf(smiles, output_filename=None):
    """
    Generate molecule from SMILES and calculate energy, gradient, and frequencies with PySCF.

    Args:
        smiles (str): SMILES string of the molecule
        output_filename (str, optional): Filename to save results

    Returns:
        dict: Dictionary containing calculation results
    """
    print(f"Input SMILES: {smiles}")

    # Create molecule from SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")

    # Get molecule name for output
    mol_name = Chem.MolToSmiles(mol)

    # Generate 3D coordinates
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol) == -1:
        print("Warning: Failed to generate 3D coordinates. Trying with random coordinates.")
        AllChem.EmbedMolecule(mol, useRandomCoords=True)

    # Optimize with MMFF
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"Error during MMFF optimization: {e}")

    # Extract atom coordinates
    conf = mol.GetConformer()
    atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    atom_coords = [[conf.GetAtomPosition(atom.GetIdx()).x,
                    conf.GetAtomPosition(atom.GetIdx()).y,
                    conf.GetAtomPosition(atom.GetIdx()).z] for atom in mol.GetAtoms()]

    print(f"Molecule contains {len(atom_symbols)} atoms")

    # Create PySCF molecule object
    atom_data = [[symbol, coord] for symbol, coord in zip(atom_symbols, atom_coords)]
    pyscf_mol = gto.Mole()
    pyscf_mol.atom = atom_data
    pyscf_mol.basis = '6-31g(d)'
    pyscf_mol.charge = 0
    pyscf_mol.spin = 0  # singlet
    pyscf_mol.verbose = 2
    pyscf_mol.build()

    # Run DFT calculation
    mf = dft.RKS(pyscf_mol)
    mf.xc = 'B3LYP'

    try:
        mf.kernel()
        print(f"SCF energy: {mf.e_tot:.8f} Hartree")
        scf_success = True
    except Exception as e:
        print(f"SCF calculation error: {e}")
        return {"success": False, "error": str(e)}

    # Calculate gradient
    try:
        g = grad.RKS(mf)
        gradients = g.kernel()
        grad_norm = np.linalg.norm(gradients)
        print(f"Gradient norm: {grad_norm:.6f}")
        grad_success = True
    except Exception as e:
        print(f"Gradient calculation error: {e}")
        grad_success = False
        grad_norm = None

    # Calculate Hessian for vibrational analysis
    freq_cm1 = None
    n_imag = None
    try:
        hess = hessian.RHF(mf)
        h = hess.kernel()

        # Convert Hessian from (natm,natm,3,3) to (3*natm,3*natm)
        natm = pyscf_mol.natm
        h_mat = np.zeros((3*natm, 3*natm))
        for i in range(natm):
            for j in range(natm):
                for k in range(3):
                    for l in range(3):
                        h_mat[i*3+k, j*3+l] = h[i, j, k, l]

        # Atomic masses
        masses = {
            'H': 1.008, 'C': 12.011, 'O': 15.999, 'N': 14.007,
            'F': 18.998, 'Cl': 35.453, 'Br': 79.904, 'I': 126.904,
            'S': 32.065, 'P': 30.974, 'Si': 28.085, 'B': 10.811
        }

        # Create mass-weighted Hessian
        atom_masses = [masses.get(atom[0], 1.0) for atom in pyscf_mol.atom]
        mass_mat = np.zeros((3*natm, 3*natm))
        for i, mass in enumerate(atom_masses):
            for j in range(3):
                mass_mat[3*i+j, 3*i+j] = 1.0 / np.sqrt(mass)

        mw_hess = np.dot(np.dot(mass_mat, h_mat), mass_mat)

        # Calculate frequencies
        eigvals, eigvecs = np.linalg.eigh(mw_hess)
        hartree_to_cm1 = 219474.6
        freq_cm1 = np.sign(eigvals) * np.sqrt(np.abs(eigvals)) * hartree_to_cm1

        # Count imaginary frequencies
        n_imag = np.sum(freq_cm1 < 0)
        if n_imag > 0:
            print(f"Warning: {n_imag} imaginary frequencies found")
        else:
            print("All frequencies are positive. Structure is stable.")

        hess_success = True
    except Exception as e:
        print(f"Hessian calculation error: {e}")
        hess_success = False

    # Set default output filename if none provided
    if output_filename is None:
        output_filename = f"{mol_name.replace('/', '_')}_pyscf_results.txt"

    # Save results to file
    with open(output_filename, 'w') as f:
        f.write(f"{mol_name} PySCF Calculation Results\n\n")
        f.write("Method: B3LYP/6-31G(d)\n\n")

        f.write("Molecular structure (RDKit optimized):\n")
        for symbol, coord in zip(atom_symbols, atom_coords):
            f.write(f"{symbol} {coord[0]:.6f} {coord[1]:.6f} {coord[2]:.6f}\n")

        f.write(f"\nSCF energy: {mf.e_tot:.8f} Hartree\n")
        if grad_success:
            f.write(f"Gradient norm: {grad_norm:.6f}\n\n")

        if hess_success and freq_cm1 is not None:
            f.write("Vibrational frequencies (cm-1):\n")
            for i, freq in enumerate(freq_cm1):
                f.write(f"Mode {i+1}: {freq:.2f} cm-1\n")

            if n_imag > 0:
                f.write(f"\nWarning: {n_imag} imaginary frequencies found.\n")
            else:
                f.write("\nAll frequencies are positive. Structure is stable.\n")

    print(f"Results saved to '{output_filename}'")

    # Return results dictionary
    return {
        "success": True,
        "molecule": mol_name,
        "energy": mf.e_tot,
        "gradient_norm": grad_norm if grad_success else None,
        "frequencies": freq_cm1.tolist() if hess_success and freq_cm1 is not None else None,
        "output_file": output_filename
    }

def main():
    """Run PySCF calculation from user-provided SMILES string"""
    print("PySCF calculation from SMILES string")
    print("====================================")

    # Get user input
    smiles = input("Enter SMILES code: ")
    output_filename = input("Enter output filename (default: [molecule]_pyscf_results.txt): ")

    if output_filename.strip() == "":
        output_filename = None

    # Run calculation
    try:
        results = calculate_with_pyscf(smiles, output_filename)
        if results["success"]:
            print("\nCalculation completed successfully")
        else:
            print(f"\nCalculation failed: {results.get('error', 'Unknown error')}")
    except Exception as e:
        print(f"\nUnexpected error: {e}")

if __name__ == "__main__":
    main()

PySCF calculation from SMILES string
Enter SMILES code: CC
Enter output filename (default: [molecule]_pyscf_results.txt): ethane
Input SMILES: CC
Molecule contains 8 atoms
SCF energy: -79.82854245 Hartree
Gradient norm: 0.010182
Results saved to 'ethane'

Calculation completed successfully


In [ ]:
## Program to search molecules by name in PubChem and calculate properties using PySCF

import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from pyscf import gto, dft, grad, hessian
import time

def search_and_calculate(molecule_name, output_filename=None):
    """
    Search for molecule by name in PubChem and perform quantum chemistry calculations

    Args:
        molecule_name (str): Name of the molecule to search for
        output_filename (str, optional): Filename to save results

    Returns:
        dict: Dictionary containing calculation results
    """
    print(f"Input molecule name: {molecule_name}")

    # Search PubChem for the molecule
    try:
        compounds = pcp.get_compounds(molecule_name, 'name')

        if not compounds:
            print(f"No results found for '{molecule_name}'")
            return {"success": False, "error": "No search results"}

        # Get SMILES from first result
        compound = compounds[0]
        smiles = compound.isomeric_smiles
        print(f"Found compound: {compound.iupac_name}")
        print(f"SMILES: {smiles}")
        print(f"Formula: {compound.molecular_formula}")
        print(f"Weight: {compound.molecular_weight} g/mol")
        print(f"PubChem CID: {compound.cid}")

    except Exception as e:
        print(f"PubChem search error: {e}")
        return {"success": False, "error": str(e)}

    # Create molecule from SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"success": False, "error": "Failed to create RDKit molecule"}

    # Generate 3D coordinates
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol) == -1:
        print("Warning: Failed to generate 3D coordinates. Trying with random coordinates.")
        AllChem.EmbedMolecule(mol, useRandomCoords=True)

    # Optimize with MMFF
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception as e:
        print(f"MMFF optimization error: {e}")

    # Extract atom coordinates
    conf = mol.GetConformer()
    atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    atom_coords = [[conf.GetAtomPosition(atom.GetIdx()).x,
                    conf.GetAtomPosition(atom.GetIdx()).y,
                    conf.GetAtomPosition(atom.GetIdx()).z] for atom in mol.GetAtoms()]

    print(f"Molecule contains {len(atom_symbols)} atoms")

    # Create PySCF molecule
    atom_data = [[symbol, coord] for symbol, coord in zip(atom_symbols, atom_coords)]
    pyscf_mol = gto.Mole()
    pyscf_mol.atom = atom_data
    pyscf_mol.basis = '6-31g(d)'
    pyscf_mol.charge = 0
    pyscf_mol.spin = 0  # singlet
    pyscf_mol.verbose = 2

    try:
        pyscf_mol.build()
    except Exception as e:
        print(f"PySCF molecule creation error: {e}")
        return {"success": False, "error": str(e)}

    # Run DFT calculation
    mf = dft.RKS(pyscf_mol)
    mf.xc = 'B3LYP'

    try:
        mf.kernel()
        print(f"SCF energy: {mf.e_tot:.8f} Hartree")
        scf_success = True
    except Exception as e:
        print(f"SCF calculation error: {e}")
        return {"success": False, "error": str(e)}

    # Calculate gradient
    try:
        g = grad.RKS(mf)
        gradients = g.kernel()
        grad_norm = np.linalg.norm(gradients)
        print(f"Gradient norm: {grad_norm:.6f}")
        grad_success = True
    except Exception as e:
        print(f"Gradient calculation error: {e}")
        grad_success = False
        grad_norm = None

    # Calculate Hessian for vibrational analysis
    freq_cm1 = None
    n_imag = None
    try:
        hess = hessian.RHF(mf)
        h = hess.kernel()

        # Convert Hessian from (natm,natm,3,3) to (3*natm,3*natm)
        natm = pyscf_mol.natm
        h_mat = np.zeros((3*natm, 3*natm))
        for i in range(natm):
            for j in range(natm):
                for k in range(3):
                    for l in range(3):
                        h_mat[i*3+k, j*3+l] = h[i, j, k, l]

        # Define atomic masses dictionary
        masses = {
            'H': 1.008, 'C': 12.011, 'O': 15.999, 'N': 14.007,
            'F': 18.998, 'Cl': 35.453, 'Br': 79.904, 'I': 126.904,
            'S': 32.065, 'P': 30.974, 'Si': 28.085, 'B': 10.811
        }

        # Get atomic masses
        atom_masses = [masses.get(atom[0], 1.0) for atom in pyscf_mol.atom]

        # Create mass matrix
        mass_mat = np.zeros((3*natm, 3*natm))
        for i, mass in enumerate(atom_masses):
            for j in range(3):
                mass_mat[3*i+j, 3*i+j] = 1.0 / np.sqrt(mass)

        # Calculate mass-weighted Hessian
        mw_hess = np.dot(np.dot(mass_mat, h_mat), mass_mat)

        # Calculate frequencies
        eigvals, eigvecs = np.linalg.eigh(mw_hess)
        hartree_to_cm1 = 219474.6
        freq_cm1 = np.sign(eigvals) * np.sqrt(np.abs(eigvals)) * hartree_to_cm1

        # Count imaginary frequencies
        n_imag = np.sum(freq_cm1 < 0)
        if n_imag > 0:
            print(f"Warning: {n_imag} imaginary frequencies found")
        else:
            print("All frequencies are positive. Structure is stable.")

        hess_success = True
    except Exception as e:
        print(f"Hessian calculation error: {e}")
        hess_success = False

    # Set default output filename if none provided
    if output_filename is None:
        output_filename = f"{molecule_name.replace(' ', '_')}_pyscf_results.txt"

    # Save results to file
    with open(output_filename, 'w') as f:
        f.write(f"{molecule_name} PySCF Calculation Results\n")
        f.write("==================================\n\n")
        f.write(f"PubChem Information:\n")
        f.write(f"- IUPAC Name: {compound.iupac_name}\n")
        f.write(f"- SMILES: {smiles}\n")
        f.write(f"- Formula: {compound.molecular_formula}\n")
        f.write(f"- Molecular Weight: {compound.molecular_weight} g/mol\n")
        f.write(f"- PubChem CID: {compound.cid}\n\n")

        f.write("Method: B3LYP/6-31G(d)\n\n")

        f.write("Molecular Structure (RDKit optimized):\n")
        for symbol, coord in zip(atom_symbols, atom_coords):
            f.write(f"{symbol} {coord[0]:.6f} {coord[1]:.6f} {coord[2]:.6f}\n")

        f.write(f"\nSCF Energy: {mf.e_tot:.8f} Hartree\n")
        if grad_success:
            f.write(f"Gradient Norm: {grad_norm:.6f}\n\n")

        if hess_success and freq_cm1 is not None:
            f.write("Vibrational Frequencies (cm-1):\n")
            for i, freq in enumerate(freq_cm1):
                f.write(f"Mode {i+1}: {freq:.2f} cm-1\n")

            if n_imag and n_imag > 0:
                f.write(f"\nWarning: {n_imag} imaginary frequencies found.\n")
            else:
                f.write("\nAll frequencies are positive. Structure is stable.\n")

    print(f"Results saved to '{output_filename}'")

    # Return results dictionary
    return {
        "success": True,
        "molecule_name": molecule_name,
        "iupac_name": compound.iupac_name,
        "smiles": smiles,
        "formula": compound.molecular_formula,
        "weight": compound.molecular_weight,
        "pubchem_cid": compound.cid,
        "energy": mf.e_tot,
        "gradient_norm": grad_norm if grad_success else None,
        "frequencies": freq_cm1.tolist() if hess_success and freq_cm1 is not None else None,
        "imaginary_frequencies": int(n_imag) if n_imag is not None else None,
        "output_file": output_filename
    }

def main():
    """Run PubChem search and PySCF calculation based on user input"""
    print("PubChem Molecular Search and PySCF Calculation")
    print("=============================================")

    # Get user input
    molecule_name = input("Enter molecule name: ")
    output_filename = input("Enter output filename (default: [molecule_name]_pyscf_results.txt): ")

    if output_filename.strip() == "":
        output_filename = None

    # Run calculation
    try:
        results = search_and_calculate(molecule_name, output_filename)
        if results["success"]:
            print("\nCalculation completed successfully")
        else:
            print(f"\nCalculation failed: {results.get('error', 'Unknown error')}")
    except Exception as e:
        print(f"\nUnexpected error: {e}")

if __name__ == "__main__":
    main()

PubChem Molecular Search and PySCF Calculation
Enter molecule name: methanol
Enter output filename (default: [molecule_name]_pyscf_results.txt): methanol
Input molecule name: methanol
Found compound: methanol
SMILES: CO
Formula: CH4O
Weight: 32.042 g/mol
PubChem CID: 887
Molecule contains 6 atoms
SCF energy: -115.71114552 Hartree
Gradient norm: 0.018667
Results saved to 'methanol'

Calculation completed successfully
